# Deploy our agent locally to Studio and to LangGraph Cloud.

Concepts
There are a few central concepts to understand -

LangGraph —

Python and JavaScript library
Allows creation of agent workflows
LangGraph API —

Bundles the graph code
Provides a task queue for managing asynchronous operations
Offers persistence for maintaining state across interactions
LangSmith Deployment (formerly LangGraph Cloud) --

Hosted service for the LangGraph API
Allows deployment of graphs from GitHub repositories
Also provides monitoring and tracing for deployed graphs
Accessible via a unique URL for each deployment
LangSmith Studio (formerly LangGraph Studio) --

Integrated Development Environment (IDE) for LangGraph applications
Uses the API as its back-end, allowing real-time testing and exploration of graphs
Can be run locally or with cloud-deployment. See below.

LangGraph SDK --

Python library for programmatically interacting with LangGraph graphs
Provides a consistent interface for working with graphs, whether served locally or in the cloud
Allows creation of clients, access to assistants, thread management, and execution of runs

### Testing Locally

LanSmith studio can now be run locally and accessed through your browser. Browser is the preferred way to run Studio instead of using the Desktop App.
To start the local development server, run the following command in your terminal in the /studio directory in this module:
BAsh
```
langgraph dev
```
You should see the following output:
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
Open your browser and navigate to the Studio UI URL shown above.
```
NOTE: Dont forget to git bash langgraph dev

In [5]:
# use langraph sdk for connection
from langgraph_sdk import get_client

# This is the URL of the local development server
url = "http://127.0.0.1:2024"
client = get_client(url = url)


## Search all the hosted graphs

In [6]:
assistants = await client.assistants.search()
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-04-17T17:59:57.156516+00:00',
  'updated_at': '2026-04-17T17:59:57.156516+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': 'b836ac90-f4c8-5b21-9bb7-c3a740fb334a',
  'graph_id': 'tool_call_router',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'tool_call_router',
  'created_at': '2026-04-17T17:59:37.306074+00:00',
  'updated_at': '2026-04-17T17:59:37.306074+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '77e0c272-e435-5999-8d0a-efea8c68be26',
  'graph_id': 'mood_graph_2',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'mood_graph_2',
  'created_at': '2026-04-17T17:59:24.854482+00:00',
  'updated_at': '2026-04-17T17:59:24.854482+00:00',
  'version': 1,
  'description': None},
 {'assistant_

In [7]:
assistants[0]

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'config': {},
 'context': {},
 'metadata': {'created_by': 'system'},
 'name': 'agent',
 'created_at': '2026-04-17T17:59:57.156516+00:00',
 'updated_at': '2026-04-17T17:59:57.156516+00:00',
 'version': 1,
 'description': None}

## Create a thread for memory checkpoint

Now, we can run our agent with client.runs.stream with:

The thread_id
The graph_id
The input
The stream_mode
For now, just recognize that we are streaming the full value of the state after each step of the graph with stream_mode="values"
The state is captured in the chunk.data.

In [14]:
''' We create a thread for tracking the state of our run This returns:

python
{"thread_id": "abc123"}
'''
thread = await client.threads.create()

In [12]:
from langchain_core.messages import HumanMessage

msg_input = {'messages': [HumanMessage(content= "Add 30 to 20")]}

# stream

async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=msg_input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Add 30 to 20', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '183bffd5-69e0-4bbd-a700-48ee8c35b400'}
{'content': [], 'additional_kwargs': {'function_call': {'name': 'addition', 'arguments': '{"b": 20, "a": 30}'}, '__gemini_function_call_thought_signatures__': {'e130f812-7e0d-4678-8f33-89c3387f5271': 'EjQKMgEMOdbHVlACw2X9+YtyKOMnJrLECJ8dPFs+cw8p5N1nRpTRUvmGdQ/ygrbsS94g02IJ'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019d9cb0-b2e0-71a3-9df6-f3d527b9c212-0', 'tool_calls': [{'name': 'addition', 'args': {'b': 20, 'a': 30}, 'id': 'e130f812-7e0d-4678-8f33-89c3387f5271', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 254, 'output_tokens': 18, 'total_tokens': 272, 'input_token_details': {'cache_read': 0}}}
{'content': '50.0', 'additional_kwargs': {}, 're

In [13]:
msg_input = {'messages': [HumanMessage(content= "now multiply that with 2")]}

# stream

async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=msg_input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'now multiply that with 2', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '52520caf-8051-4cac-952a-aed914f3a481'}
{'content': [], 'additional_kwargs': {'function_call': {'name': 'multiply', 'arguments': '{"b": 2, "a": 50}'}, '__gemini_function_call_thought_signatures__': {'ba681f25-f550-4ade-bce5-be6924d4d08c': 'EjQKMgEMOdbHEDo+JK5tl+G4qfhxSDG9H/heMFAC6YgpQ07Zox/wp7jWpQS+0GDNKQ9GnXAE'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019d9cb2-1dc5-7b60-a09c-5faf811b93ec-0', 'tool_calls': [{'name': 'multiply', 'args': {'b': 2, 'a': 50}, 'id': 'ba681f25-f550-4ade-bce5-be6924d4d08c', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 307, 'output_tokens': 17, 'total_tokens': 324, 'input_token_details': {'cache_read': 0}}}
{'content': '100.0', 'additional_kwarg

## Test with cloud